# 🧾 Extrator de NFS-e - Brasil
## Sistema de Extração, Validação e Consolidação de Notas Fiscais

Este notebook permite processar múltiplas NFS-e em PDFs e gerar uma planilha Excel consolidada.

---

## 📦 1. Instalação de Dependências

In [ ]:
!pip install -q pdfplumber pandas openpyxl
print("✅ Dependências instaladas com sucesso!")

## 📥 2. Upload dos Arquivos do Sistema

Execute a célula abaixo para fazer upload dos arquivos Python do sistema:
- `extractor.py`
- `validator.py`
- `consolidator.py`

In [ ]:
from google.colab import files
import os

print("📤 Faça upload dos arquivos .py do sistema (extractor.py, validator.py, consolidator.py)")
uploaded = files.upload()

print(f"\n✅ {len(uploaded)} arquivo(s) carregado(s)")
for filename in uploaded.keys():
    print(f"   • {filename}")

## 📄 3. Upload das Notas Fiscais (PDFs)

Faça upload de todos os PDFs de NFS-e que deseja processar:

In [ ]:
from google.colab import files
import os

# Cria diretório para os PDFs
os.makedirs('notas_fiscais', exist_ok=True)

print("📤 Faça upload dos PDFs de NFS-e")
uploaded_pdfs = files.upload()

# Move PDFs para o diretório
for filename in uploaded_pdfs.keys():
    if filename.lower().endswith('.pdf'):
        os.rename(filename, f'notas_fiscais/{filename}')

pdf_count = len([f for f in os.listdir('notas_fiscais') if f.endswith('.pdf')])
print(f"\n✅ {pdf_count} arquivo(s) PDF carregado(s)")

## 🔄 4. Processamento das Notas Fiscais

Esta célula irá:
1. Extrair dados de todos os PDFs
2. Validar as informações
3. Gerar planilha Excel consolidada

In [ ]:
from extractor import NFSeExtractor
from validator import NFSeValidator
from consolidator import NFSeConsolidator
from pathlib import Path

# Inicializa componentes
print("🚀 Iniciando processamento...\n")
extractor = NFSeExtractor()
validator = NFSeValidator()
consolidator = NFSeConsolidator()

# Encontra PDFs
pdf_files = list(Path('notas_fiscais').glob('*.pdf'))
print(f"📄 Encontrados {len(pdf_files)} arquivo(s) PDF\n")

# Extrai dados
print("📊 Extraindo dados...")
extracted_data = []
errors = []

for i, pdf_path in enumerate(pdf_files, 1):
    try:
        print(f"   [{i}/{len(pdf_files)}] {pdf_path.name}", end='')
        data = extractor.extract_from_pdf(str(pdf_path))
        extracted_data.append(data)
        print(f" ✓ NF {data.numero_nota} - R$ {data.valor_servicos}")
    except Exception as e:
        errors.append(f"{pdf_path.name}: {str(e)}")
        print(f" ✗ Erro")

print(f"\n✅ Extraídos {len(extracted_data)} nota(s)")

if errors:
    print(f"⚠️  {len(errors)} erro(s):")
    for error in errors[:3]:
        print(f"    • {error}")

# Valida dados
print("\n🔍 Validando dados...")
validation_counts = {'ok': 0, 'warning': 0, 'error': 0}

for data in extracted_data:
    issues = validator.validate(data)
    if any(i.severity == 'ERROR' for i in issues):
        validation_counts['error'] += 1
    elif any(i.severity == 'WARNING' for i in issues):
        validation_counts['warning'] += 1
    else:
        validation_counts['ok'] += 1

print(f"   ✅ Validadas: {validation_counts['ok']}")
print(f"   ⚠️  Com avisos: {validation_counts['warning']}")
print(f"   ❌ Com erros: {validation_counts['error']}")

# Gera planilha
print("\n📊 Gerando planilha consolidada...")
output_file = 'nfse_consolidado.xlsx'
consolidator.consolidate_to_excel(extracted_data, output_file, include_validation=True)

# Resumo
total_valor = sum(float(d.valor_servicos) for d in extracted_data)
total_iss = sum(float(d.iss_valor) for d in extracted_data)
total_tributos = sum(float(d.total_tributos_retidos) for d in extracted_data)

print("\n" + "="*60)
print("RESUMO EXECUTIVO")
print("="*60)
print(f"Notas processadas:      {len(extracted_data)}")
print(f"Valor total:            R$ {total_valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
print(f"ISS retido:             R$ {total_iss:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
print(f"Total tributos retidos: R$ {total_tributos:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
print("="*60)

print(f"\n✅ Planilha gerada: {output_file}")

## 📥 5. Download da Planilha Consolidada

Baixe a planilha Excel gerada:

In [ ]:
from google.colab import files

print("📥 Iniciando download da planilha...")
files.download('nfse_consolidado.xlsx')
print("✅ Download concluído!")

## 📊 6. Visualização Rápida dos Dados

Visualize uma prévia dos dados extraídos:

In [ ]:
import pandas as pd

# Carrega planilha
df = pd.read_excel('nfse_consolidado.xlsx', sheet_name='Dados NFS-e')

print("📊 DADOS EXTRAÍDOS (primeiras 5 notas):\n")
display(df.head())

print("\n📈 ESTATÍSTICAS:")
print(f"Total de notas: {len(df)}")
print(f"Valor total: R$ {df['Valor Serviços'].sum():,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))
print(f"ISS total: R$ {df['ISS Valor'].sum():,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

print("\n🏛️ DISTRIBUIÇÃO POR MUNICÍPIO:")
display(df.groupby('Município')['Valor Serviços'].agg(['count', 'sum']).round(2))

---

## 💡 Dicas de Uso

1. **Múltiplos Uploads**: Você pode executar a célula de upload várias vezes para adicionar mais PDFs
2. **Reprocessamento**: Para processar novamente, execute as células 4 e 5
3. **Validação**: A planilha "Validação" contém detalhes de todos os problemas encontrados
4. **Resumo**: A planilha "Resumo" tem totalizadores e estatísticas

## 🆘 Problemas?

- Abra uma issue no GitHub: https://github.com/RAFAELSOUZA280292/ExtractorNFSe/issues
- Verifique se todos os arquivos .py foram carregados
- Certifique-se de que os PDFs são de NFS-e válidas

---

**Desenvolvido para facilitar o trabalho de contadores e profissionais financeiros** 💼